# Analysis 1 — Retention Segmentation
**Phase 3 | Quick Commerce Operations Project**

Business questions answered here:
- What drives repeat purchases?
- Which customer segments retain best?

Definition: repeat customer = `customer_unique_id` with 2+ orders (any category)

Input: `workspace.default.final_quick_comm_dataset` as `df`  
Outputs: Tableau-ready CSVs in `/outputs/`

## 0. Imports & Setup

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings, os
warnings.filterwarnings('ignore')
os.makedirs('outputs', exist_ok=True)

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   '#f9f9f7',
    'axes.spines.top':  False,
    'axes.spines.right':False,
    'axes.grid':        True,
    'grid.alpha':       0.4,
    'grid.linestyle':   '--',
    'font.size':        11,
})
PALETTE = ['#3B8BD4','#1D9E75','#F2A623','#E8593C','#7F77DD','#888780']
print('Setup complete.')

## 1. Data Preparation

In [0]:
# Deduplicate to one row per order
# customer_unique_id persists across orders; customer_id does not
df = spark.table(
    "workspace.default.final_quick_comm_dataset"
).toPandas()

date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_cols:
    df[col] = pd.to_datetime(df[col])
    
orders = (
    df
    .drop_duplicates(subset='order_id', keep='first')
    [['order_id','customer_unique_id','order_purchase_timestamp',
      'order_status','payment_value','review_score']]
    .copy()
)

# Keep only delivered orders for retention analysis
orders = orders[orders['order_status'] == 'delivered'].copy()

# Parse dates
orders['order_purchase_timestamp'] = pd.to_datetime(
    orders['order_purchase_timestamp'])

# Cohort month = month of customer's FIRST order
orders['order_month'] = orders['order_purchase_timestamp'].dt.to_period('M')

first_order = (
    orders
    .groupby('customer_unique_id')['order_purchase_timestamp']
    .min()
    .reset_index()
    .rename(columns={'order_purchase_timestamp': 'first_order_date'})
)
first_order['cohort_month'] = first_order['first_order_date'].dt.to_period('M')

orders = orders.merge(first_order[['customer_unique_id','cohort_month']],
                      on='customer_unique_id', how='left')

# Period index: how many months after acquisition did this order happen?
orders['period_index'] = (
    (orders['order_month'] - orders['cohort_month'])
    .apply(lambda x: x.n)
)

print(f'Delivered orders : {len(orders):,}')
print(f'Unique customers : {orders["customer_unique_id"].nunique():,}')
print(f'Date range       : {orders["order_purchase_timestamp"].min().date()} '
      f'to {orders["order_purchase_timestamp"].max().date()}')

## 2. Repeat Customer KPIs

In [0]:
order_counts = (
    orders
    .groupby('customer_unique_id')['order_id']
    .count()
    .reset_index()
    .rename(columns={'order_id': 'total_orders'})
)

total_customers  = len(order_counts)
repeat_customers = (order_counts['total_orders'] >= 2).sum()
repeat_rate      = repeat_customers / total_customers * 100
avg_orders       = order_counts['total_orders'].mean()

print('── Retention KPIs ───────────────────────────────')
print(f'  Total customers         : {total_customers:,}')
print(f'  Repeat customers (2+)   : {repeat_customers:,}')
print(f'  Repeat rate             : {repeat_rate:.1f}%')
print(f'  Avg orders per customer : {avg_orders:.2f}')

# Order frequency distribution
freq_dist = (
    order_counts['total_orders']
    .clip(upper=5)   # cap at 5+ for clean chart
    .value_counts()
    .sort_index()
)
freq_dist.index = [str(i) if i < 5 else '5+' for i in freq_dist.index]

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(freq_dist.index, freq_dist.values / total_customers * 100,
              color=PALETTE, edgecolor='white', linewidth=0.8, width=0.6)
for bar, val in zip(bars, freq_dist.values / total_customers * 100):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{val:.1f}%', ha='center', va='bottom', fontsize=10)
ax.set_xlabel('Number of orders per customer')
ax.set_ylabel('% of customers')
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
ax.set_title(f'Order frequency distribution  |  Repeat rate = {repeat_rate:.1f}%', pad=12)
plt.tight_layout()
plt.savefig('outputs/fig_order_frequency.png', dpi=150, bbox_inches='tight')
plt.show()

# Export KPIs
pd.DataFrame([{
    'total_customers':  total_customers,
    'repeat_customers': int(repeat_customers),
    'repeat_rate_pct':  round(repeat_rate, 1),
    'avg_orders_per_customer': round(avg_orders, 2),
}]).to_csv('outputs/retention_kpis.csv', index=False)

## 3. Cohort Retention Table
Each row = acquisition cohort (month of first order).  
Each column = months since acquisition (period index).  
Values = % of cohort customers who placed at least one order in that month.

In [0]:
# Count distinct active customers per cohort x period
cohort_data = (
    orders
    .groupby(['cohort_month', 'period_index'])['customer_unique_id']
    .nunique()
    .reset_index()
    .rename(columns={'customer_unique_id': 'active_customers'})
)

# Cohort sizes (period 0 = acquisition month)
cohort_sizes = (
    cohort_data[cohort_data['period_index'] == 0]
    .set_index('cohort_month')['active_customers']
    .rename('cohort_size')
)

# Pivot to matrix
cohort_pivot = cohort_data.pivot_table(
    index='cohort_month',
    columns='period_index',
    values='active_customers'
)

# Convert to retention % relative to cohort size
retention = cohort_pivot.divide(cohort_sizes, axis=0) * 100

# Keep only period indices 0-11 (first 12 months)
max_period = min(11, retention.columns.max())
retention  = retention.loc[:, 0:max_period]
retention.columns = [f'Month {i}' for i in retention.columns]
retention.index   = retention.index.astype(str)

# Drop cohorts with fewer than 30 customers (too small to be meaningful)
retention = retention[cohort_sizes >= 30]

print(f'Cohort table: {retention.shape[0]} cohorts x {retention.shape[1]} periods')
print(retention.round(1).to_string())

In [0]:
# ── Heatmap
import matplotlib.colors as mcolors

fig, ax = plt.subplots(figsize=(14, max(5, len(retention) * 0.45)))

# Custom colormap: white -> teal
cmap = mcolors.LinearSegmentedColormap.from_list(
    'retention', ['#ffffff', '#9FE1CB', '#1D9E75', '#085041'])

im = ax.imshow(retention.values, aspect='auto', cmap=cmap,
               vmin=0, vmax=100)

# Annotate each cell
for i in range(retention.shape[0]):
    for j in range(retention.shape[1]):
        val = retention.values[i, j]
        if not np.isnan(val):
            color = 'white' if val > 55 else '#2C2C2A'
            ax.text(j, i, f'{val:.1f}%', ha='center', va='center',
                    fontsize=8, color=color)

ax.set_xticks(range(retention.shape[1]))
ax.set_xticklabels(retention.columns, rotation=30, ha='right', fontsize=9)
ax.set_yticks(range(retention.shape[0]))
ax.set_yticklabels(retention.index, fontsize=9)
ax.set_xlabel('Months since first order')
ax.set_ylabel('Acquisition cohort')
ax.set_title('Monthly cohort retention  |  % of cohort active each month', pad=12)

plt.colorbar(im, ax=ax, label='Retention %', shrink=0.6)
plt.tight_layout()
plt.savefig('outputs/fig_cohort_retention.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Month 1 Retention Deep-Dive
Month 1 retention (did the customer return the following month?) is the strongest leading indicator of long-term retention.

In [0]:
if 'Month 1' in retention.columns:
    m1 = retention['Month 1'].dropna().sort_values(ascending=False)

    fig, ax = plt.subplots(figsize=(9, max(4, len(m1) * 0.38)))
    colors  = ['#1D9E75' if v >= 10 else '#F2A623' if v >= 5
               else '#E8593C' for v in m1.values]
    bars = ax.barh(m1.index, m1.values, color=colors)
    for bar, val in zip(bars, m1.values):
        ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
                f'{val:.1f}%', va='center', fontsize=9)
    ax.set_xlabel('Month 1 retention (%)')
    ax.set_xlim(0, m1.max() * 1.2)
    ax.xaxis.set_major_formatter(mticker.PercentFormatter())
    ax.set_title('Month 1 retention by acquisition cohort', pad=12)
    ax.invert_yaxis()
    plt.tight_layout()
    plt.savefig('outputs/fig_month1_retention.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Month 1 column not available — dataset may not have enough repeat orders.')

## 5. One-time vs Repeat Customer Segment Profiles

In [0]:
# Tag each customer
order_counts['segment'] = np.where(
    order_counts['total_orders'] >= 2, 'Repeat', 'One-time')

# Join back to orders
orders = orders.merge(
    order_counts[['customer_unique_id','segment','total_orders']],
    on='customer_unique_id', how='left'
)

seg_profile = (
    orders
    .groupby('segment')
    .agg(
        customers       = ('customer_unique_id', 'nunique'),
        total_orders    = ('order_id',           'count'),
        avg_order_value = ('payment_value',       'mean'),
        avg_review_score= ('review_score',        'mean'),
        total_revenue   = ('payment_value',       'sum'),
    )
    .reset_index()
)
seg_profile['avg_order_value']  = seg_profile['avg_order_value'].round(2)
seg_profile['avg_review_score'] = seg_profile['avg_review_score'].round(2)
seg_profile['revenue_share_pct']= (
    seg_profile['total_revenue'] / seg_profile['total_revenue'].sum() * 100
).round(1)

print('Segment profiles:')
print(seg_profile.to_string(index=False))
seg_profile.to_csv('outputs/segment_profiles.csv', index=False)

# ── Side-by-side bars: AOV and review score by segment
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
metrics = [
    ('avg_order_value',  'Avg order value (R$)', 'R${:.0f}'),
    ('avg_review_score', 'Avg review score',     '{:.2f}'),
    ('revenue_share_pct','Revenue share (%)',    '{:.1f}%'),
]
for ax, (col, title, fmt) in zip(axes, metrics):
    bars = ax.bar(seg_profile['segment'], seg_profile[col],
                  color=['#3B8BD4','#1D9E75'], width=0.5,
                  edgecolor='white', linewidth=0.8)
    for bar, val in zip(bars, seg_profile[col]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                fmt.format(val), ha='center', va='bottom', fontsize=10)
    ax.set_title(title, pad=8)
    ax.set_ylim(0, seg_profile[col].max() * 1.25)

plt.suptitle('One-time vs repeat customer segment profiles', y=1.02, fontsize=12)
plt.tight_layout()
plt.savefig('outputs/fig_segment_profiles.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Which Categories Retain Best?
For repeat customers only — what category did they re-order in?

In [0]:
repeat_orders = orders[orders['segment'] == 'Repeat'].copy()

# Merge category from original df
repeat_orders = repeat_orders.merge(
    df[['order_id','product_category_name_english']].drop_duplicates('order_id'),
    on='order_id', how='left'
)

cat_retention = (
    repeat_orders
    .groupby('product_category_name_english')
    .agg(
        repeat_orders    = ('order_id',           'count'),
        repeat_customers = ('customer_unique_id',  'nunique'),
        avg_order_value  = ('payment_value',       'mean'),
    )
    .reset_index()
)
cat_retention = cat_retention[cat_retention['repeat_orders'] >= 30].copy()
cat_retention['avg_order_value'] = cat_retention['avg_order_value'].round(2)
cat_retention = cat_retention.sort_values('repeat_customers', ascending=False)

print('Top 10 categories by repeat customer count:')
print(cat_retention.head(10).to_string(index=False))
cat_retention.to_csv('outputs/category_retention.csv', index=False)

# Chart
top10 = cat_retention.head(10)
fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(top10['product_category_name_english'],
               top10['repeat_customers'], color='#3B8BD4')
for bar, val in zip(bars, top10['repeat_customers']):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
            f'{val:,}', va='center', fontsize=9)
ax.set_xlabel('Number of repeat customers')
ax.set_title('Top 10 categories by repeat customer count\n(min 30 repeat orders)', pad=12)
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('outputs/fig_category_retention.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Export Cohort Table for Tableau

In [0]:
# Wide format (heatmap in Tableau)
retention.round(2).to_csv('outputs/cohort_retention_wide.csv')

# Long format (easier to build line charts in Tableau)
retention_long = (
    retention
    .reset_index()
    .melt(id_vars='cohort_month', var_name='period', value_name='retention_pct')
    .dropna(subset=['retention_pct'])
)
retention_long['period_num'] = (
    retention_long['period'].str.replace('Month ', '').astype(int)
)
retention_long.to_csv('outputs/cohort_retention_long.csv', index=False)

print('Exported:')
print('  outputs/cohort_retention_wide.csv  — heatmap source')
print('  outputs/cohort_retention_long.csv  — line chart source')

## 8. Output Summary

| File | Tableau section |
|------|----------------|
| `retention_kpis.csv` | Section 1 — Executive KPIs |
| `cohort_retention_wide.csv` | Section 4 — Customer Retention (heatmap) |
| `cohort_retention_long.csv` | Section 4 — Customer Retention (line chart) |
| `segment_profiles.csv` | Section 4 — Customer Retention |
| `category_retention.csv` | Section 3 — Categories |

**Key questions answered:**
- *What drives repeat purchases?* See segment profiles — compare AOV and review score between one-time and repeat customers
- *Which customer segments retain best?* See cohort heatmap — look for cohorts with sustained Month 1+ retention

In [0]:
df = spark.table(
    "workspace.default.final_quick_comm_dataset"
).toPandas()

date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]


In [0]:
df.to_csv('outputs/final_quick_comm_dataset.csv', index=False)